# 小红书关键词抓取 Demo（基于 Scrapling PoC）

本 Notebook 支持：
- 单个关键词 / CSV 批量关键词
- `api` 与 `render` 双模式
- 关键词之间随机等待 5~6 分钟（降低风控触发概率）
- 每个关键词单独 CSV + 合并 CSV 输出

> 如果你追求“翻页翻到底”的可控性，推荐优先使用 `api` 模式（依赖 `has_more` 分页）。


In [ ]:
import random
import time
import sys
import importlib.util
from pathlib import Path
from typing import List

import pandas as pd

module_path = Path('examples/xhs_keyword_poc.py').resolve()
spec = importlib.util.spec_from_file_location('xhs_keyword_poc', module_path)
xhs_mod = importlib.util.module_from_spec(spec)
assert spec and spec.loader, f'Cannot load module from: {module_path}'
sys.modules[spec.name] = xhs_mod
spec.loader.exec_module(xhs_mod)

run_api_mode = getattr(xhs_mod, 'run_api_mode', None) or getattr(xhs_mod, 'run_api')
run_render_mode = getattr(xhs_mod, 'run_render_mode', None) or getattr(xhs_mod, 'run_render')


In [ ]:
# ===== 用户配置 =====
MODE = 'api'  # 'api' or 'render'

# 单关键词模式（CSV_KEYWORDS_PATH=None 时生效）
SINGLE_KEYWORD = '穿搭'

# CSV 批量关键词模式
CSV_KEYWORDS_PATH = None  # 例如：'keywords.csv'
CSV_KEYWORD_COLUMN = 'keyword'

COOKIE_HEADER = ''

# api 模式参数
API_HEADERS_RAW = '{}'
SEARCH_API_URL = 'https://edith.xiaohongshu.com/api/sns/web/v1/search/notes'
DETAIL_API_URL = 'https://edith.xiaohongshu.com/api/sns/web/v1/feed'
API_PAGE_SIZE = 20

# render 模式参数
RENDER_SCROLL_ROUNDS = 120
RENDER_IDLE_ROUNDS = 3

# 通用参数
MAX_NOTES_PER_KEYWORD = 100000
OUTPUT_DIR = Path('outputs/xhs_demo')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_keywords(single_keyword: str, csv_path: str | None, keyword_col: str = 'keyword') -> List[str]:
    if csv_path is None:
        return [single_keyword.strip()] if single_keyword.strip() else []

    df = pd.read_csv(csv_path)
    if keyword_col not in df.columns:
        raise ValueError(f'CSV 中未找到列: {keyword_col}, 可用列: {df.columns.tolist()}')

    keywords = (
        df[keyword_col]
        .dropna()
        .astype(str)
        .str.strip()
        .replace('', pd.NA)
        .dropna()
        .tolist()
    )
    return list(dict.fromkeys(keywords))  # 去重保序


def sleep_5_to_6_minutes():
    sec = random.uniform(300, 360)
    print(f'⏳ Sleep {sec:.1f}s (~{sec/60:.2f} min) ...')
    time.sleep(sec)


def run_one_keyword(keyword: str):
    if MODE == 'api':
        return run_api_mode(
            keyword=keyword,
            max_notes=MAX_NOTES_PER_KEYWORD,
            cookie_header=COOKIE_HEADER,
            api_headers_raw=API_HEADERS_RAW,
            search_api_url=SEARCH_API_URL,
            detail_api_url=DETAIL_API_URL,
            page_size=API_PAGE_SIZE,
        )

    return run_render_mode(
        keyword=keyword,
        max_notes=MAX_NOTES_PER_KEYWORD,
        cookie_header=COOKIE_HEADER,
        scroll_rounds=RENDER_SCROLL_ROUNDS,
        render_idle_rounds=RENDER_IDLE_ROUNDS,
    )


In [ ]:
keywords = load_keywords(SINGLE_KEYWORD, CSV_KEYWORDS_PATH, CSV_KEYWORD_COLUMN)
print(f'Loaded {len(keywords)} keyword(s): {keywords[:10]}')

all_frames = []
for idx, kw in enumerate(keywords, start=1):
    print(f'\n===== [{idx}/{len(keywords)}] {kw} ({MODE}) =====')
    t0 = time.time()

    items = run_one_keyword(kw)
    rows = []
    for item in items:
        row = item.__dict__.copy()
        row['keyword'] = kw
        rows.append(row)

    df_kw = pd.DataFrame(rows)
    kw_file = OUTPUT_DIR / f'{idx:03d}_{kw}_notes.csv'
    df_kw.to_csv(kw_file, index=False, encoding='utf-8-sig')
    all_frames.append(df_kw)

    print(f'✅ rows={len(df_kw)} -> {kw_file}')
    print(f'⏱ elapsed={time.time()-t0:.1f}s')

    if idx < len(keywords):
        sleep_5_to_6_minutes()

merged = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
merged_file = OUTPUT_DIR / 'all_keywords_merged.csv'
merged.to_csv(merged_file, index=False, encoding='utf-8-sig')
print(f'\n🎉 merged rows={len(merged)} -> {merged_file}')


## CSV 示例

`keywords.csv` 例如：

| keyword |
|---|
| 穿搭 |
| 护肤 |
| 旅行攻略 |
